# Stage 3: SOAP vs SOAP + MoQ

This notebook runs the Stage 3 experiments for PINN Poisson optimization, comparing the canonical SOAP optimizer with and without Momentum of Quality (MoQ).

In [ ]:
import json
import math
import time
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Ensure you have official_soap.py in your working directory
try:
    from official_soap import SOAP
except ImportError:
    import sys
    # If uploaded as a dataset in kaggle, you might need to append the path:
    # sys.path.append('/kaggle/input/your-dataset-name')
    from official_soap import SOAP


In [ ]:
class PINN(nn.Module):
    def __init__(self, width: int = 128, hidden_layers: int = 4) -> None:
        super().__init__()
        layers: List[nn.Module] = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(width, width), nn.Tanh()])
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, xy: torch.Tensor) -> torch.Tensor:
        return self.net(xy)

def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [ ]:
def sample_collocation(n: int, device: torch.device) -> torch.Tensor:
    return torch.rand(n, 2, device=device, requires_grad=True)

def sample_boundary(n_each: int, device: torch.device) -> torch.Tensor:
    t = torch.rand(4 * n_each, device=device)
    x = torch.empty(4 * n_each, 2, device=device)
    x[:n_each] = torch.stack([torch.zeros(n_each, device=device), t[:n_each]], dim=1)
    x[n_each:2*n_each] = torch.stack([torch.ones(n_each, device=device), t[n_each:2*n_each]], dim=1)
    x[2*n_each:3*n_each] = torch.stack([t[2*n_each:3*n_each], torch.zeros(n_each, device=device)], dim=1)
    x[3*n_each:] = torch.stack([t[3*n_each:], torch.ones(n_each, device=device)], dim=1)
    return x

def make_batch_schedule(steps: int, K: int, n_pde: int, n_boundary_each: int, device: torch.device) -> List[Dict[str, torch.Tensor]]:
    schedule: List[Dict[str, torch.Tensor]] = []
    pde: Optional[torch.Tensor] = None
    bc: Optional[torch.Tensor] = None

    for step in range(1, steps + 1):
        if step == 1 or (step - 1) % K == 0:
            pde = sample_collocation(n_pde, device)
            bc = sample_boundary(n_boundary_each, device)
        schedule.append({
            "pde": pde.detach().clone(),
            "bc": bc.detach().clone(),
        })
    return schedule


In [ ]:
def poisson_residual(model: nn.Module, xy: torch.Tensor) -> torch.Tensor:
    u = model(xy)
    grad_u = torch.autograd.grad(u, xy, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]
    u_x, u_y = grad_u[:, 0:1], grad_u[:, 1:2]
    u_xx = torch.autograd.grad(u_x, xy, grad_outputs=torch.ones_like(u_x), create_graph=True, retain_graph=True)[0][:, 0:1]
    u_yy = torch.autograd.grad(u_y, xy, grad_outputs=torch.ones_like(u_y), create_graph=True, retain_graph=True)[0][:, 1:2]
    forcing = 2.0 * math.pi**2 * torch.sin(math.pi * xy[:, 0:1]) * torch.sin(math.pi * xy[:, 1:2])
    return -(u_xx + u_yy) - forcing

def relative_l2(model: nn.Module, device: torch.device, n: int = 81) -> float:
    grid = torch.linspace(0.0, 1.0, n, device=device)
    xx, yy = torch.meshgrid(grid, grid, indexing="ij")
    xy = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
    with torch.no_grad():
        pred = model(xy).reshape(-1)
    ref = torch.sin(math.pi * xy[:, 0]) * torch.sin(math.pi * xy[:, 1])
    return float(torch.linalg.vector_norm(pred - ref) / (torch.linalg.vector_norm(ref) + 1e-12))


In [ ]:
def flat_grad(model: nn.Module) -> torch.Tensor:
    return torch.cat([p.grad.detach().reshape(-1) for p in model.parameters() if p.grad is not None])

def get_flat_params(model: nn.Module) -> torch.Tensor:
    return torch.cat([p.detach().reshape(-1) for p in model.parameters()])

def set_flat_grad(model: nn.Module, flat_g: torch.Tensor) -> None:
    offset = 0
    for p in model.parameters():
        numel = p.numel()
        if p.grad is not None:
            p.grad.data.copy_(flat_g[offset:offset+numel].reshape(p.shape))
        offset += numel


In [ ]:
def run_experiment(K: int, steps: int, use_moq: bool, mu: float = 0.85, seed: int = 0):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = PINN().to(device)
    
    # Load or create schedule
    schedule_path = Path(f"poisson_seed{seed}_K{K}_schedule.pt")
    if schedule_path.exists():
        print(f"Loading existing schedule: {schedule_path}")
        schedule = torch.load(schedule_path, map_location=device, weights_only=True)
    else:
        print(f"Creating new schedule: {schedule_path}")
        schedule = make_batch_schedule(max(2000, steps), K, 1000, 100, device)
        torch.save(schedule, schedule_path)

    # Canonical SOAP initialization
    lr = 1e-3
    opt = SOAP(model.parameters(), lr=lr, betas=(0.95, 0.95), weight_decay=0.0, precondition_frequency=10)

    prev_grad: Optional[torch.Tensor] = None
    history: List[Dict[str, float]] = []

    model.train()
    for step in range(1, steps + 1):
        current_pde = schedule[step - 1]["pde"].to(device).detach().clone().requires_grad_(True)
        current_bc = schedule[step - 1]["bc"].to(device)
        opt.zero_grad(set_to_none=True)

        r = poisson_residual(model, current_pde)
        loss_pde = torch.mean(r.square())
        u_bc = model(current_bc)
        loss_bc = torch.mean(u_bc.square())
        loss = loss_pde + loss_bc
        loss.backward()

        g_t = flat_grad(model)
        grad_norm = float(torch.linalg.vector_norm(g_t))
        
        grad_diff_norm = float("nan")
        grad_change_ratio = float("nan")
        grad_cosine = float("nan")
        MoQ_gradient_norm = float("nan")
        MoQ_raw_gradient_norm = float("nan")
        cos_gt_gMoQ = float("nan")
        
        g_opt = g_t.clone()
        
        if prev_grad is not None:
            diff = g_t - prev_grad
            grad_diff_norm = float(torch.linalg.vector_norm(diff))
            grad_change_ratio = grad_diff_norm / (grad_norm + 1e-12)
            grad_cosine = float(torch.dot(g_t, prev_grad) / (grad_norm * torch.linalg.vector_norm(prev_grad) + 1e-12))
            
            g_MoQ = g_t + mu * diff
            MoQ_gradient_norm = float(torch.linalg.vector_norm(g_MoQ))
            MoQ_raw_gradient_norm = MoQ_gradient_norm / (grad_norm + 1e-12)
            cos_gt_gMoQ = float(torch.dot(g_t, g_MoQ) / (grad_norm * MoQ_gradient_norm + 1e-12))
            
            if use_moq:
                g_opt = g_MoQ
        else:
            MoQ_gradient_norm = grad_norm
            MoQ_raw_gradient_norm = 1.0
            cos_gt_gMoQ = 1.0

        if use_moq:
            set_flat_grad(model, g_opt)
            
        params_before = get_flat_params(model)
        opt.step()
        params_after = get_flat_params(model)
        
        parameter_update_norm = float(torch.linalg.vector_norm(params_after - params_before))
        preconditioned_update_norm = parameter_update_norm / lr
        
        prev_grad = g_t.clone() # Always store actual raw gradient

        if step == 1 or step % 50 == 0 or step == steps:
            err = relative_l2(model, device)
            print(f"step={step:4d} loss={float(loss):.4e} rel_l2={err:.4e}")
        else:
            err = relative_l2(model, device)

        history.append({
            "step": step,
            "loss": float(loss.detach()),
            "rel_l2": err,
            "grad_norm": grad_norm,
            "grad_diff_norm": grad_diff_norm,
            "grad_change_ratio": grad_change_ratio,
            "grad_cosine": grad_cosine,
            "MoQ_gradient_norm": MoQ_gradient_norm,
            "MoQ/raw_gradient_norm": MoQ_raw_gradient_norm,
            "preconditioned_update_norm": preconditioned_update_norm,
            "parameter_update_norm": parameter_update_norm,
            "cos(g_t, g_MoQ)": cos_gt_gMoQ
        })

    return history


### 1. SOAP vs SOAP+MoQ with K=1

In [ ]:
print("Running SOAP (K=1, 500 steps)...")
history_soap_K1 = run_experiment(K=1, steps=500, use_moq=False)
with open("stage3_soap_K1_500.json", "w") as f:
    json.dump({"K": 1, "use_moq": False, "history": history_soap_K1}, f, indent=2)


In [ ]:
print("Running SOAP+MoQ (K=1, 500 steps, mu=0.85)...")
history_moq_K1 = run_experiment(K=1, steps=500, use_moq=True, mu=0.85)
with open("stage3_soap_moq_K1_500.json", "w") as f:
    json.dump({"K": 1, "use_moq": True, "mu": 0.85, "history": history_moq_K1}, f, indent=2)


### 2. SOAP vs SOAP+MoQ with K=20

In [ ]:
print("Running SOAP (K=20, 500 steps)...")
history_soap_K20 = run_experiment(K=20, steps=500, use_moq=False)
with open("stage3_soap_K20_500.json", "w") as f:
    json.dump({"K": 20, "use_moq": False, "history": history_soap_K20}, f, indent=2)


In [ ]:
print("Running SOAP+MoQ (K=20, 500 steps, mu=0.85)...")
history_moq_K20 = run_experiment(K=20, steps=500, use_moq=True, mu=0.85)
with open("stage3_soap_moq_K20_500.json", "w") as f:
    json.dump({"K": 20, "use_moq": True, "mu": 0.85, "history": history_moq_K20}, f, indent=2)
